# Train the Mamba crack segmenter on Google Colab

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU), then run the cells from top to bottom. The dataset archive is included in this repository. Training outputs are saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
!git clone https://github.com/1234usamas/mamba.git /content/mamba
%cd /content/mamba
!pip install -q -r requirements.txt -e .

In [ ]:
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('A GPU is not enabled. In Colab, select Runtime → Change runtime type → GPU.')

dataset_dir = Path('/content/mamba/under-crack-images')
if not (dataset_dir / 'images').exists() or not (dataset_dir / 'labels').exists():
    subprocess.run(['unzip', '-q', '-o', '/content/mamba/under-crack-images.zip', '-d', str(dataset_dir)], check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Images:', len(list((dataset_dir / 'images').glob('*'))))
print('Labels:', len(list((dataset_dir / 'labels').glob('*'))))

## Train

The default run trains up to 150 epochs and stops early based on validation clDice. Epoch tables include train and validation loss, Dice, clDice, and Betti number error. Checkpoints, CSV results, and TensorBoard logs are written to Drive.

In [ ]:
%cd /content/mamba
!python train.py --config config.yaml --output-dir /content/drive/MyDrive/mamba_segmenter

## Evaluate the best checkpoint

This writes test metrics and visualizations next to the checkpoint in Drive.

In [ ]:
%cd /content/mamba
!python evaluate.py --checkpoint /content/drive/MyDrive/mamba_segmenter/best.pt